# 09 · Verbinding en uitwisseling tussen Droplets controleren

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Vanaf notebook 05 wissel je via internet meetgegevens uit tussen twee afzonderlijke DigitalOcean-Droplets. Als je thuis of op een ander tijdstip werkt dan de andere gebruiker, wil je snel kunnen nagaan of alles goed staat ingesteld, waarom een meting nog niet aankomt en wat de ander moet doen als een server of broker stilstaat.

Dit hulphandboek vat samen:
1. Welke gegevens je via chat deelt en waarvoor elk onderdeel dient.
2. Aan welke voorwaarde beide servers vooraf moeten voldoen (en waarom je niet tegelijk achter de laptop hoeft te zitten).
3. Waarom een gewone `ping` misleidend is en hoe je vanuit Codespaces wél test of jouw Droplet en die van de andere gebruiker via TLS-MQTT (`8883`) en HTTPS (`443`) bereikbaar zijn.
4. Welke exacte opdrachten de andere gebruiker (of jij) op de Droplet uitvoert om Mosquitto en `iot-ontvanger` weer actief te krijgen.

Je kunt dit notebook op elk moment vanaf notebook 03 gebruiken, ook als notebook 05, 06 of 08 nog niet is afgerond.

</div>


## 1. Wat wissel je uit via chat, en waarvoor dient elk detail?

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

De twee servers praten niet rechtstreeks met elkaar als gelijke brokers. Elke Droplet heeft een eigen **ontvanger** (`iot-ontvanger`) die met twee verbindingen tegelijk leest: één keer bij je eigen Mosquitto-broker (`eigenaar`) en één keer bij de Mosquitto-broker van de andere gebruiker (`lezer`).

In **notebook 05** wisselen twee gebruikers met een tegengestelde rol (`gewichten` ↔ `voederopname`) één keer onderstaande gegevens uit via chat of e-mail:

| Gegeven | Waar vind je het? | Functie bij de uitwisseling | Mag je dit delen? |
|---|---|---|---|
| **Deelnemerscode** | `01_werkplekcontrole.json` | Vormt het exacte topic `practicum/<code >/metingen/<rol>` en controleert het veld `deelnemer` in elk JSON-bericht. | Ja |
| **Rol** | `gewichten` of `voederopname` | Koppelt `gewichten` aan `asserva` en `voederopname` aan `schauer`, zodat je altijd twee verschillende stromen ontvangt. | Ja |
| **Publiek IPv4-adres** | `gegevens/werk/server.json` | Het internetadres van de andere Droplet waarmee jouw `iot-ontvanger` op TCP-poort `8883` verbindt. | Ja |
| **Bestand `ca.crt` + SHA-256-vingerafdruk** | `gegevens/werk/ca.crt` | Het openbare certificaat van de andere server (opgeslagen als `ca_partner.crt`). Bewijst tijdens de TLS-verbinding dat je echt met die Droplet praat. | Ja (`ca.crt` wel, **`ca.key` en `server.key` nooit**) |
| **Wachtwoord `MQTT-broker lezer`** | Wachtwoordmanager (uit notebook 03) | Alleen-lezen login (`read practicum/#`) voor de andere ontvanger. Geeft géén schrijfrechten en géén toegang tot SSH of API. | Alleen via besloten bericht aan je partner |
| **`05_broncontrole.json`** | `gegevens/inlevering/` (na notebook 05) | Bevat alleen aantallen en SHA-256-hashes (geen wachtwoorden), om te bewijzen dat alle records van je partner volledig zijn aangekomen. | Ja |

In **notebook 08** hoef je de IP-adressen, certificaten en wachtwoorden **niet opnieuw** uit te wisselen als beide Droplets sinds notebook 05 blijven doorlopen. Dan stuur je alleen **`08_eigen_broncontrole.json`** door (de ander bewaart dat als `gegevens/werk/08_partner_broncontrole.json`), zodat notebook 08 de verwachte nieuwe recordaantallen kent.

</div>

<div style="background-color:#fff8e1;color:#6d4c00;border-left:5px solid #f9a825;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;white-space:normal;overflow-wrap:anywhere;"><strong>Deel nooit deze geheime gegevens</strong><br>Stuur nooit jouw <strong><code>SSH beheer-wachtwoord</code></strong>, <strong><code>MQTT-broker eigenaar</code></strong>, <strong><code>API-Droplet wachtwoord</code></strong>, <code>server.key</code>, <code>ca.key</code> of <code>/etc/iot/ontvangst.geheim.json</code> naar een andere gebruiker en zet ze nooit in Git.</div>


## 2. Voorwaarde en timing: moeten we tegelijk online zijn?

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Nee**, jullie hoeven niet in hetzelfde lokaal te zitten en ook niet op exact dezelfde seconde op *Run* te drukken. De Droplets blijven immers dag en nacht aanstaan, ook nadat je jouw laptop of Codespace hebt gesloten.

Toch is er **één harde technische voorwaarde** voordat iemand metingen publiceert:

1. **De ontvanger van je partner moet al geabonneerd zijn vóórdat jij verstuurt.** In deze opzet stuurt Mosquitto berichten rechtstreeks door naar ontvangers die op dat moment verbonden en geabonneerd zijn (`ABONNEMENT GEREED`). Berichten die jij verstuurt terwijl de andere `iot-ontvanger` nog niet ingesteld is of uitstaat, worden door de broker **niet achteraf nagestuurd**.
2. **Beide relevante services moeten draaien tijdens het versturen:** bij de verzender moet **`mosquitto`** draaien (`active`) en bij de ontvanger moet **`iot-ontvanger`** draaien (`active`, met twee keer `gereed`).

**Twee veelvoorkomende valkuilen uit eerdere notebooks:**
- **Notebook 04 eindigt met `sudo systemctl stop mosquitto`.** Wie daarna in notebook 05 vergeet `sudo systemctl start mosquitto` uit te voeren, heeft een onbereikbare broker.
- **Na een herstart van de Droplet start `mosquitto` niet vanzelf mee.** De service staat bewust op `disabled`, terwijl `nginx`, `iot-status` en `iot-ontvanger` wél vanzelf starten.

**Wat als iemand toch te vroeg heeft gepubliceerd?**  
Dat is zonder dataverlies te herstellen. Zodra de ontvanger op de andere Droplet wel op `ABONNEMENT GEREED` staat, hoeft de verzender in notebook 05 of 08 alleen zijn publicatiecel (`publiceer_beginset`) opnieuw uit te voeren. De database ontdubbelt alle al opgeslagen records automatisch op `(bron, record_id)`.

</div>


## 3. Waarom `ping` misleidend is (en hoe je vanuit Codespaces wél test)

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Studenten proberen soms `ping <IP-adres>` om te zien of de Droplet van een partner bereikbaar is. Dat helpt in dit practicum maar zelden:
- Vanuit een container zoals GitHub Codespaces wordt ICMP-verkeer (`ping`) vaak geblokkeerd of tegengehouden, waardoor `ping` ook bij een perfect werkende Droplet niets teruggeeft.
- Zelfs als `ping` wel zou antwoorden, bewijst dat alleen dat de virtuele machine aanstaat, niet dat **Mosquitto** (`8883`) draait of dat de TLS-certificaten kloppen.

In de cellen hieronder test Python daarom rechtstreeks de twee echte practicumpoorten over het netwerk:
- **TCP-poort `8883` + TLS-handdruk** tegen Mosquitto (met `ca.crt` voor je eigen Droplet en `ca_partner.crt` voor je partner).
- **TCP-poort `443` + HTTPS-aanvraag op `/gezondheid`** tegen Nginx en `iot-status`.

Een groot voordeel van die combinatie is dat je meteen het verschil ziet: antwoordt poort `443` met HTTP `401` of `200`, maar wordt poort `8883` geweigerd, dan **staat de Droplet zeker aan** en staat alleen **Mosquitto** nog uit!

</div>


In [ ]:
_DOCENT_BRANCH_OVERRIDE = "student/proefdocent"


In [ ]:
from pathlib import Path
import hashlib
import ipaddress
import json
import re
import socket
import ssl
import subprocess
import requests

r = subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True)
if r.returncode:
    raise RuntimeError("Open notebook 09 vanuit de practicumrepository in Codespaces.")
hoofdmap = Path(r.stdout.strip())
branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=hoofdmap, text=True).strip()
if not re.fullmatch(r"student/[a-z0-9][a-z0-9_-]*", branch) and "_DOCENT_BRANCH_OVERRIDE" in globals():
    branch = _DOCENT_BRANCH_OVERRIDE

werkmap = hoofdmap / "gegevens/werk"
inlevermap = hoofdmap / "gegevens/inlevering"

def lees_json_indien_aanwezig(pad: Path):
    return json.loads(pad.read_text(encoding="utf-8")) if pad.is_file() else None

eigen_server = lees_json_indien_aanwezig(werkmap / "server.json")
eigen_ca = werkmap / "ca.crt"
partner_server = lees_json_indien_aanwezig(werkmap / "partner.json")
partner_ca = werkmap / "ca_partner.crt"
instellingen = lees_json_indien_aanwezig(inlevermap / "01_werkplekcontrole.json")

print("Branch:", branch)
if instellingen:
    print("Eigen deelnemerscode:", instellingen.get("deelnemerscode"), "| Rol:", instellingen.get("rol"))
print("Eigen server.json:", eigen_server["ipv4"] if eigen_server else "ontbreekt (rond eerst notebook 03 af)")
print("Eigen ca.crt:", "aanwezig" if eigen_ca.is_file() else "ontbreekt")
print("Partner partner.json:", f"{partner_server['deelnemerscode']} ({partner_server['ipv4']}, {partner_server['rol']})" if partner_server else "nog niet ingevuld (sectie 5 van notebook 05)")
print("Partner ca_partner.crt:", "aanwezig" if partner_ca.is_file() else "nog niet geüpload")


In [ ]:
def test_poort_en_tls(naam: str, ip_tekst: str, ca_pad: Path | None):
    ip = str(ipaddress.IPv4Address(ip_tekst.strip()))
    print(f"=== Controle voor {naam} ({ip}) ===")

    # 1. HTTPS (poort 443): bewijst of de Droplet aanstaat en nginx/iot-status draaien.
    poort_443_ok = False
    try:
        with socket.create_connection((ip, 443), timeout=5):
            poort_443_ok = True
        print("  [OK]   TCP 443 (HTTPS) is bereikbaar.")
    except OSError as fout:
        print(f"  [FOUT] TCP 443 (HTTPS) onbereikbaar: {fout}")

    # 2. MQTT over TLS (poort 8883): bewijst of Mosquitto draait en bereikbaar is.
    poort_8883_ok = False
    try:
        with socket.create_connection((ip, 8883), timeout=5):
            poort_8883_ok = True
        print("  [OK]   TCP 8883 (MQTT) is bereikbaar.")
    except OSError as fout:
        print(f"  [FOUT] TCP 8883 (MQTT) onbereikbaar: {fout}")

    # 3. TLS-certificaatcontrole op 8883 en 443 (zonder wachtwoord).
    if ca_pad and ca_pad.is_file():
        sha = hashlib.sha256(ca_pad.read_bytes()).hexdigest()
        print(f"  [INFO] SHA-256 van {ca_pad.name}: {sha}")
        if poort_8883_ok:
            try:
                ctx = ssl.create_default_context(cafile=str(ca_pad))
                ctx.minimum_version = ssl.TLSVersion.TLSv1_2
                with socket.create_connection((ip, 8883), timeout=5) as rauw:
                    with ctx.wrap_socket(rauw, server_hostname=ip):
                        pass
                print("  [OK]   TLS-handdruk op poort 8883 slaagt met dit CA-bestand.")
            except ssl.SSLError as fout:
                print(f"  [FOUT] TLS-controle op poort 8883 mislukt: {fout}")
                print("         Controleer of je het juiste ca.crt / ca_partner.crt voor dit IPv4-adres gebruikt.")
        if poort_443_ok:
            try:
                r = requests.get(f"https://{ip}/gezondheid", verify=str(ca_pad), timeout=5)
                if r.status_code in (200, 401):
                    print(f"  [OK]   HTTPS-endpoint /gezondheid antwoordt met HTTP {r.status_code} (nginx + iot-status zijn actief).")
                else:
                    print(f"  [LET OP] HTTPS gaf HTTP {r.status_code} op /gezondheid.")
            except requests.exceptions.SSLError as fout:
                print(f"  [FOUT] HTTPS TLS-controle mislukt: {fout}")
            except requests.exceptions.RequestException as fout:
                print(f"  [FOUT] HTTPS-verzoek mislukt: {fout}")
    else:
        print("  [INFO] Geen CA-bestand meegegeven; TLS-handdruk overgeslagen.")

    # 4. Korte samenvattende diagnose
    if poort_443_ok and not poort_8883_ok:
        print("  --> DIAGNOSE: De Droplet staat WEL aan (443 werkt), maar Mosquitto staat UIT of poort 8883 is geblokkeerd.")
        print("      Laat op die Droplet via SSH uitvoeren: sudo systemctl start mosquitto")
    elif not poort_443_ok and not poort_8883_ok:
        print("  --> DIAGNOSE: De Droplet is volledig onbereikbaar (uitgezet, verkeerd IPv4-adres of DigitalOcean-firewall blokkeert).")
    elif poort_443_ok and poort_8883_ok:
        print("  --> DIAGNOSE: Netwerk en beide serverpoorten zijn bereikbaar.")


In [ ]:
if eigen_server:
    test_poort_en_tls("Eigen Droplet", eigen_server["ipv4"], eigen_ca)
else:
    print("Geen eigen gegevens/werk/server.json gevonden.")

print()
if partner_server:
    test_poort_en_tls(f"Partner ({partner_server['deelnemerscode']})", partner_server["ipv4"], partner_ca)
else:
    print("Geen gegevens/werk/partner.json gevonden. Vul hieronder eventueel tijdelijk een IP-adres in om handmatig te testen:")
    # Voorbeeld voor handmatige test zonder partner.json:
    # test_poort_en_tls("Partner handmatig", "203.0.113.10", partner_ca if partner_ca.is_file() else None)


<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Als poort `8883` en de TLS-handdruk hierboven slagen, maar `iot-ontvanger` toch **`aanmelding_geweigerd`** meldt, ligt het meestal aan het MQTT-wachtwoord of aan `/etc/mosquitto/practicum.rechten`. Met de hulpfunctie in de cel hieronder kun je (alleen wanneer je hem handmatig aanroept) één korte MQTT-verbindingstest uitvoeren zonder berichten te sturen.

</div>


In [ ]:
import sys
from getpass import getpass
sys.path[:0] = [str(hoofdmap / "programmatuur")]
from publiceren import MqttSessie

def test_mqtt_login(ip_tekst: str, ca_pad: Path, gebruiker: str, topic: str):
    wachtwoord = getpass(f"Wachtwoord voor MQTT-gebruiker '{gebruiker}' op {ip_tekst}: ")
    try:
        with MqttSessie(ip_tekst, ca_pad, wachtwoord, abonnement=topic) as sessie:
            print(f"[OK] Verbonden en geabonneerd als '{gebruiker}' op {sessie.bestemming} ({topic}).")
    finally:
        del wachtwoord

# Verwijder het #-teken bij één van onderstaande regels als je een wachtwoord wilt testen:
# test_mqtt_login(eigen_server["ipv4"], eigen_ca, "eigenaar", f"practicum/{instellingen['deelnemerscode']}/metingen/{instellingen['rol']}")
# test_mqtt_login(partner_server["ipv4"], partner_ca, "lezer", f"practicum/{partner_server['deelnemerscode']}/metingen/{partner_server['rol']}")


## 4. Wat doe je (of je partner) als de server of Mosquitto uitstaat?

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Let goed op **waar** je een opdracht uitvoert:
- **`Codespaces-terminal`** (prompt begint met `root@codespaces-...`): alleen om met `ssh` in te loggen, bestanden met `scp` te kopiëren of Git en notebooks te draaien.
- **`SSH-terminal op de Droplet`** (prompt begint met `beheer@...`): alleen daar draaien `mosquitto`, `iot-ontvanger`, `iot-status` en `nginx`, en alleen daar werken `systemctl` en `journalctl`.

</div>

### Stap A. Inloggen op de Droplet (uitvoeren: in Codespaces-terminal)

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Open in Codespaces een terminal en log in op de eigen Droplet met het **`SSH beheer-wachtwoord`** (vervang `JOUW_SERVER_IP` door het eigen IPv4-adres uit `gegevens/werk/server.json`):

</div>

```bash
ssh -o HostKeyAlgorithms=ssh-ed25519 beheer@JOUW_SERVER_IP
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

*(Geeft `ssh` zelf een time-out? Kijk dan op DigitalOcean in project `starten-met-iot` of jouw Droplet niet uitgezet is en zet hem daar weer aan.)*

</div>

### Stap B. Mosquitto en `iot-ontvanger` weer starten en controleren (uitvoeren: op de Droplet via SSH, als `beheer`)

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Voer deze regels uit op de Droplet zodra de prompt `beheer@...` toont. Je start Mosquitto handmatig (omdat die na een herstart uitstaat), herstart `iot-ontvanger` en controleert alle vier de diensten:

</div>

```bash
sudo systemctl start mosquitto
sudo systemctl restart iot-ontvanger
sudo systemctl status mosquitto iot-ontvanger nginx iot-status --no-pager
```

### Stap C. Controleren of beide abonnementen op `gereed` staan (uitvoeren: op de Droplet via SSH, als `beheer`)

```bash
sudo journalctl -u iot-ontvanger -n 25 --no-pager
sudo -u iot /opt/iot/venv/bin/python /opt/iot/programmatuur/ontvangst_status.py
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Je verwacht in het logboek twee regels **`ABONNEMENT GEREED`** (één voor je eigen bron en één voor je partner) en in het statusoverzicht twee keer `toestand: "gereed"`.

</div>

### Stap D. Partnerinstellingen of firewall herstellen als `iot-ontvanger` blijft falen (uitvoeren: op de Droplet via SSH)

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

- Zie je in het logboek **`aanmelding_geweigerd`** of een verkeerd IP-adres? Laat de ander eerst bevestigen dat `mosquitto` draait en zijn/haar **`MQTT-broker lezer`**-wachtwoord klopt, en werk de partnerinstelling bij zonder je opgeslagen metingen kwijt te raken:

</div>

```bash
sudo /opt/iot/venv/bin/python /opt/iot/programmatuur/ontvangst_instellen.py --partner --vervang
sudo systemctl restart iot-ontvanger
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

- Controleer bij poortfouten of alle drie de poorten (`22`, `443`, `8883`) in de firewall openstaan en beluisterd worden:

</div>

```bash
sudo ufw status numbered
sudo ss -lntp
```

<div style="background-color:#e8f5e9;color:#1b5e20;border-left:5px solid #2e7d32;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;white-space:normal;overflow-wrap:anywhere;"><strong>Daarna opnieuw aanbieden (alleen als metingen gemist zijn)</strong><br>Zodra op de ontvangende Droplet beide stromen weer op <code>ABONNEMENT GEREED</code> staan, kan de andere student zijn publicatiecel in notebook 05 of 08 opnieuw draaien. Reeds ontvangen records worden netjes overgeslagen en alleen ontbrekende records worden toegevoegd.</div>


## 5. Snelle foutentabel

| Waarneming | Meest waarschijnlijke oorzaak | Waar en hoe oplossen |
|---|---|---|
| `ls ~/practicum-server: No such file or directory` | Je staat nog in je Codespaces-terminal (`root@codespaces-...`) in plaats van op je Droplet (`beheer@...`). | Log eerst in met `ssh -o HostKeyAlgorithms=ssh-ed25519 beheer@JOUW_IP` en voer de serveropdracht daar uit. |
| Poort `443` werkt, poort `8883` geweigerd | Droplet staat aan, maar `mosquitto` staat uit (bijvoorbeeld na notebook 04 of na een Droplet-herstart). | Eigenaar van die Droplet draait via SSH `sudo systemctl start mosquitto`. |
| Zowel `443` als `8883` geeft time-out | Droplet is uitgezet, IPv4-adres in `server.json`/`partner.json` klopt niet, of DigitalOcean-firewall blokkeert. | Controleer DigitalOcean dashboard (Power On) en `sudo ufw status numbered` op de Droplet. |
| `TLS-controle mislukt` / `SSLCertVerificationError` | Verkeerd `ca.crt` of `ca_partner.crt`, of het bestand hoort niet bij dit IP-adres. | Vergelijk de SHA-256-vingerafdruk van `ca.crt`, kopieer het juiste bestand opnieuw en controleer met notebook 05 stap 5. |
| `aanmelding_geweigerd` in `iot-ontvanger` | Verkeerde `MQTT-broker lezer`-login of wachtwoord, of foutieve `deelnemerscode` / `rol`. | Voer op de ontvangende Droplet `ontvangst_instellen.py --partner --vervang` uit met het juiste wachtwoord. |
| Aantallen in `05` of `08` te laag bij één van beiden | De verzender publiceerde voordat `iot-ontvanger` op de ontvangende Droplet op `ABONNEMENT GEREED` stond. | Zorg voor `ABONNEMENT GEREED` op de ontvanger en laat de verzender de publicatiecel opnieuw uitvoeren. |
